In [15]:
import os
import re
import gc
import sys
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import openslide
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# ====== PLACEHOLDER ======
# cartella dove si trovano mil_modules.py / mil_inference.py / mil_main.py
PROJECT_CODE_DIR = "/opt/app/user/postdoc/virchow_code"

if PROJECT_CODE_DIR not in sys.path:
    sys.path.append(PROJECT_CODE_DIR)

import mil_modules as mm

print("Imported mil_modules from:", mm.__file__)

Imported mil_modules from: /opt/app/user/postdoc/virchow_code/mil_modules.py


In [16]:
# ====== PLACEHOLDERS ======
CSV_PATH = "/opt/app/user/cnio_features.csv"

FEATURES_DIR = "/opt/app/user/cnio_features/2026-03-11_11_25/features"      # contains {slide_id}.pt
COORDS_DIR   = "/opt/app/user/cnio_features/2026-03-11_11_25/coordinates"        # contains {slide_id}.npy structured arrays
OUT_DIR      = "/opt/app/user/cnio_attention_zones"

MODEL_PATHS = [
    "/data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f1.pt",
    "/data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f2.pt",
    "/data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f3.pt",
    "/data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f4.pt",
    "/data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f5.pt",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# binary classification
N_CLASSES = 2

# patch geometry
PATCH_LEVEL = 0          # nel tuo caso level 0 = 0.5 um/px
DEFAULT_PATCH_SIZE = 224

# visualization
VIS_LEVEL = -1           # auto
ALPHA = 0.45
DRAW_TOPK = 10
MAX_SIZE = 4096
THUMB_MAX_SIZE = 4096

# dataloader
NUM_WORKERS = 0
BATCH_SIZE = 1

print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)

DEVICE: cuda
OUT_DIR: /opt/app/user/cnio_attention_zones


In [17]:
def parse_structured_coords(coords_arr: np.ndarray):
    """
    coords_arr: structured array with fields x, y, tile_size_lv0, ...
    Returns:
        coords_xy: (N,2) int64, top-left coords at level 0
        patch_size_lv0: int
    """
    if "x" not in coords_arr.dtype.names or "y" not in coords_arr.dtype.names:
        raise ValueError("Coordinates .npy must contain fields 'x' and 'y'.")

    x = coords_arr["x"].astype(np.int64)
    y = coords_arr["y"].astype(np.int64)
    coords_xy = np.stack([x, y], axis=1)

    if "tile_size_lv0" in coords_arr.dtype.names:
        patch_size_lv0 = int(coords_arr["tile_size_lv0"][0])
    elif "target_tile_size" in coords_arr.dtype.names:
        patch_size_lv0 = int(coords_arr["target_tile_size"][0])
    else:
        patch_size_lv0 = DEFAULT_PATCH_SIZE

    return coords_xy, patch_size_lv0


class ExternalCNIODataset(Dataset):
    """
    Expects CSV with at least:
      - slide_id
      - slide_path

    Files:
      - FEATURES_DIR/{slide_id}.pt
      - COORDS_DIR/{slide_id}.npy
    """
    def __init__(self, csv_path, features_dir, coords_dir):
        self.csv_path = Path(csv_path)
        self.features_dir = Path(features_dir)
        self.coords_dir = Path(coords_dir)

        df = pd.read_csv(self.csv_path)

        required = {"patient_id", "slide_path"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"CSV missing required columns: {missing}")

        keep = []
        for _, row in df.iterrows():
            sid = str(row["patient_id"])
            f_ok = (self.features_dir / f"{sid}.pt").is_file()
            c_ok = (self.coords_dir / f"{sid}.npy").is_file()
            s_ok = os.path.exists(str(row["slide_path"]))
            keep.append(f_ok and c_ok and s_ok)

        self.df = df.loc[keep].reset_index(drop=True)
        dropped = len(df) - len(self.df)
        print(f"Kept {len(self.df)} slides, dropped {dropped} (missing features/coords/slide_path).")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        slide_id = str(row["patient_id"])
        slide_path = str(row["slide_path"])

        feats = torch.load(self.features_dir / f"{slide_id}.pt", map_location="cpu")
        if not torch.is_tensor(feats):
            raise TypeError(f"{slide_id}.pt does not contain a tensor.")
        feats = feats.float()

        coords_struct = np.load(self.coords_dir / f"{slide_id}.npy")
        coords_xy, patch_size_lv0 = parse_structured_coords(coords_struct)

        out = {
            "slide_id": slide_id,
            "slide_path": slide_path,
            "features": feats,               # (N,D)
            "coords_xy": coords_xy,          # (N,2)
            "patch_size_lv0": patch_size_lv0
        }

        # optional labels, if present
        if "Diagnose" in row.index:
            out["Diagnose"] = row["Diagnose"]

        return out


ds = ExternalCNIODataset(CSV_PATH, FEATURES_DIR, COORDS_DIR)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

Kept 47 slides, dropped 63 (missing features/coords/slide_path).


In [18]:
def initialize_weights(m: torch.nn.Module):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)


class GatedAttention(nn.Module):
    def __init__(
        self,
        input_dim: int = 2048,
        bottleneck_dim: int = 256,
        dropout: bool = False,
        n_branches: int = 1,
    ):
        super().__init__()
        self.attention_a = nn.Sequential(
            nn.Linear(input_dim, bottleneck_dim),
            nn.Tanh(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.attention_b = nn.Sequential(
            nn.Linear(input_dim, bottleneck_dim),
            nn.Sigmoid(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.attention_c = nn.Linear(bottleneck_dim, n_branches)

    def forward(self, x):
        a = self.attention_a(x)
        b = self.attention_b(x)
        att = a.mul(b)
        att = self.attention_c(att)  # (N,1)
        return att, x

In [19]:
class AttentionSingleBranch(nn.Module):
    def __init__(
        self,
        size=None,
        use_dropout: bool = False,
        n_classes: int = 2,
    ):
        super().__init__()

        if size is None:
            size = (2048, 512, 256)

        self.n_classes = n_classes

        fc = [nn.Linear(size[0], size[1]), nn.ReLU()]
        if use_dropout:
            fc.append(nn.Dropout(0.25))

        attention_net = GatedAttention(
            input_dim=size[1],
            bottleneck_dim=size[2],
            dropout=use_dropout,
            n_branches=1
        )
        fc.append(attention_net)
        self.attention_net = nn.Sequential(*fc)

        self.classifiers = nn.Linear(size[1], n_classes)
        self.apply(initialize_weights)

        self.train_losses = []
        self.val_losses = []

    def relocate(self):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.attention_net = self.attention_net.to(device)
        self.classifiers = self.classifiers.to(device)

    def _compute_attention(self, x):
        att, h = self.attention_net(x)                    # att: (N,1), h: (N,H)
        h = h.unsqueeze(0)                               # (1,N,H)
        att = att.transpose(1, 0).unsqueeze(0)           # (1,1,N)
        att_raw = att
        att_soft = F.softmax(att, dim=2)                 # (1,1,N)
        return h, att_raw, att_soft

    def forward(self, x):
        h, att_raw, att_soft = self._compute_attention(x)
        pooled = torch.bmm(att_soft, h)                  # (1,1,H)
        logits = self.classifiers(pooled).squeeze(1)     # (1,n_classes)

        info = {
            "attention_raw": att_raw,
            "attention_softmax": att_soft,
            "attention": att_raw,   # compat
        }
        return logits, info

In [20]:
# class PatchedAttentionSingleBranch(nn.Module):
#     """
#     Same state_dict layout as mil_modules.AttentionSingleBranch,
#     but returns BOTH raw attention and softmax attention cleanly.

#     Compatible keys:
#       - patch_mlp.*
#       - attention.attention_a.*
#       - attention.attention_b.*
#       - attention.attention_c.*
#       - slide_projector.*
#       - classifier.*
#     """
#     def __init__(self, size=(1280, 512, 256), use_dropout=False, n_classes=2):
#         super().__init__()

#         assert len(size) >= 3, "size must be like (input_dim, ..., hidden_dim, att_bottleneck)"
#         self.n_classes = n_classes
#         self.use_dropout = use_dropout
#         self.size = tuple(size)

#         fc_layers = []
#         for i in range(len(size) - 2):
#             fc_layers.append(nn.Linear(size[i], size[i + 1]))
#             fc_layers.append(nn.ReLU())
#             if use_dropout:
#                 fc_layers.append(nn.Dropout(0.25))
#         self.patch_mlp = nn.Sequential(*fc_layers)

#         att_input_dim = size[-2]
#         att_bottleneck = size[-1]
#         self.attention = mm.GatedAttention(
#             input_dim=att_input_dim,
#             bottleneck_dim=att_bottleneck,
#             dropout=use_dropout,
#             n_branches=1,
#         )

#         in_dim = size[0]
#         self.slide_projector = nn.Linear(att_input_dim, in_dim)
#         self.classifier = nn.Linear(in_dim, n_classes)

#         self.apply(mm.initialize_weights)

#     def relocate(self):
#         self.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

#     def _compute_attention(self, x):
#         """
#         x: (N, D_in)

#         Returns:
#             x_transformed_3d: (1, N, H)
#             att_raw_3d:       (1, 1, N)
#             att_soft_3d:      (1, 1, N)
#         """
#         x_embed = self.patch_mlp(x)               # (N, H)
#         att_raw_2d, x_transformed = self.attention(x_embed)  # (N,1), (N,H)

#         x_transformed_3d = x_transformed.unsqueeze(0)             # (1,N,H)
#         att_raw_3d = att_raw_2d.transpose(1, 0).unsqueeze(0)      # (1,1,N)
#         att_soft_3d = F.softmax(att_raw_3d, dim=2)                # (1,1,N)

#         return x_transformed_3d, att_raw_3d, att_soft_3d

#     def forward(self, x):
#         x_transformed_3d, att_raw_3d, att_soft_3d = self._compute_attention(x)

#         pooled = torch.bmm(att_soft_3d, x_transformed_3d).squeeze(1)   # (1,H)
#         slide_embedding = self.slide_projector(pooled)                  # (1,input_dim)
#         logits = self.classifier(slide_embedding)                       # (1,n_classes)

#         results_dict = {
#             "attention": att_raw_3d,               # compatibility
#             "attention_raw": att_raw_3d,
#             "attention_softmax": att_soft_3d,
#             "slide_embedding": slide_embedding.detach()
#         }
#         return logits, results_dict


# def infer_model_size_from_state_dict(state_dict):
#     """
#     Infer size tuple for PatchedAttentionSingleBranch from checkpoint.
#     Works for mil_modules.py-style checkpoints.
#     """
#     linear_keys = []
#     for k, v in state_dict.items():
#         m = re.match(r"patch_mlp\.(\d+)\.weight$", k)
#         if m:
#             linear_keys.append((int(m.group(1)), v.shape))

#     linear_keys = sorted(linear_keys, key=lambda t: t[0])
#     if not linear_keys:
#         raise ValueError("Could not infer model size: no patch_mlp.*.weight keys found.")

#     size = [linear_keys[0][1][1]]  # input dim from first Linear in_features
#     for _, shape in linear_keys:
#         size.append(shape[0])      # out_features of each patch_mlp Linear

#     # attention bottleneck
#     att_key = "attention.attention_a.0.weight"
#     if att_key not in state_dict:
#         raise ValueError(f"Could not infer attention bottleneck: missing key {att_key}")
#     att_bottleneck = state_dict[att_key].shape[0]
#     size.append(att_bottleneck)

#     n_classes = state_dict["classifier.weight"].shape[0]

#     return tuple(size), int(n_classes)


# def load_fold_models(model_paths, device="cuda"):
#     models = []
#     for i, ckpt_path in enumerate(model_paths, start=1):
#         sd = torch.load(ckpt_path, map_location="cpu")
#         size, n_classes = infer_model_size_from_state_dict(sd)

#         model = PatchedAttentionSingleBranch(
#             size=size,
#             use_dropout=False,
#             n_classes=n_classes
#         )
#         model.load_state_dict(sd, strict=True)
#         model.to(device)
#         model.eval()

#         models.append(model)
#         print(f"Loaded fold {i}: {ckpt_path}")
#         print(f"  inferred size={size}, n_classes={n_classes}")

#     return models


models = load_fold_models(MODEL_PATHS, DEVICE)

ValueError: Could not infer model size: no patch_mlp.*.weight keys found.

In [21]:
def infer_attentionsinglebranch_size_from_state_dict(state_dict):
    """
    Inferisce size=(input_dim, hidden_dim, att_bottleneck) e n_classes
    da un checkpoint di AttentionSingleBranch.
    """
    # first Linear in attention_net
    w0 = state_dict["attention_net.0.weight"]   # (hidden_dim, input_dim)
    hidden_dim = w0.shape[0]
    input_dim = w0.shape[1]

    # attention bottleneck
    wa = state_dict["attention_net.2.attention_a.0.weight"]  # (att_bottleneck, hidden_dim)
    att_bottleneck = wa.shape[0]

    # classifier
    wc = state_dict["classifiers.weight"]  # (n_classes, hidden_dim)
    n_classes = wc.shape[0]

    size = (input_dim, hidden_dim, att_bottleneck)
    return size, int(n_classes)


def load_fold_models(model_paths, device="cuda", use_dropout=False):
    models = []
    for i, ckpt_path in enumerate(model_paths, start=1):
        sd = torch.load(ckpt_path, map_location="cpu")
        size, n_classes = infer_attentionsinglebranch_size_from_state_dict(sd)

        model = AttentionSingleBranch(
            size=size,
            use_dropout=use_dropout,
            n_classes=n_classes
        )
        model.load_state_dict(sd, strict=True)
        model.to(device)
        model.eval()

        models.append(model)
        print(f"Loaded fold {i}: {ckpt_path}")
        print(f"  inferred size={size}, n_classes={n_classes}")

    return models

models = load_fold_models(MODEL_PATHS, DEVICE)

Loaded fold 1: /data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f1.pt
  inferred size=(1024, 512, 256), n_classes=1
Loaded fold 2: /data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f2.pt
  inferred size=(1024, 512, 256), n_classes=1
Loaded fold 3: /data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f3.pt
  inferred size=(1024, 512, 256), n_classes=1
Loaded fold 4: /data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f4.pt
  inferred size=(1024, 512, 256), n_classes=1
Loaded fold 5: /data/pathology/projects/pierpaolo/Experiments/best_models/regression/clipped/uni/best_model_regression_uni_f5.pt
  inferred size=(1024, 512, 256), n_classes=1


In [ ]:
def logits_to_outputs(logits: torch.Tensor):
    """
    Returns a dict with a consistent summary for CSV saving.
    Supports:
      - binary as 2 logits
      - binary as 1 logit
      - generic multiclass
    """
    logits_np = logits.detach().cpu().squeeze().numpy()

    if logits.ndim == 2 and logits.shape[1] == 2:
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
        pred_class = int(np.argmax(probs))
        return {
            "pred_class": pred_class,
            "prob_0": float(probs[0]),
            "prob_1": float(probs[1]),
            "logit_0": float(logits_np[0]),
            "logit_1": float(logits_np[1]),
        }

    if logits.ndim == 2 and logits.shape[1] == 1:
        prob1 = torch.sigmoid(logits).detach().cpu().numpy()[0, 0]
        pred_class = int(prob1 >= 0.5)
        return {
            "pred_class": pred_class,
            "prob_1": float(prob1),
            "logit_1": float(logits_np.item()),
        }

    # fallback multiclass
    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
    out = {"pred_class": int(np.argmax(probs))}
    for i, p in enumerate(probs):
        out[f"prob_{i}"] = float(p)
    return out


def save_attention_npz(npz_path, coords_xy, att_raw_1d, att_soft_1d, patch_size_lv0):
    """
    Compatible with the conventions already used in your project.
    """
    np.savez(
        npz_path,
        coords=coords_xy.astype(np.float32),
        attention_raw=att_raw_1d.astype(np.float32),
        attention=att_soft_1d.astype(np.float32),
        patch_size_lv0=np.array([patch_size_lv0], dtype=np.int32),
    )


def render_clam_like_report(
    slide_path,
    coords_xy,
    scores_raw,
    out_png,
    patch_level=0,
    patch_size=224,
    draw_topk=10,
    alpha=0.6,
    cmap_name="plasma",
    vis_level=-1,
    max_size=4096,
    combine_subplots=True,
    layout="horizontal",
    title=None,
):
    """
    Generate a PNG report for one slide.

    combine_subplots=True  (default): WSI heatmap (left) + ranked patch grid (right).
    combine_subplots=False           : WSI heatmap only — use this for paper figures.

    The colormap (cmap_name) and alpha are shared between the heatmap overlay and
    the patch-border ranking colors so everything is visually consistent.
    """
    heatmap_img, _ = mm.clam_like_heatmap(
        slide_path=slide_path,
        coords_lvl0=coords_xy,
        scores_raw=scores_raw,
        vis_level=vis_level,
        patch_level=patch_level,
        patch_size=patch_size,
        alpha=alpha,
        cmap_name=cmap_name,
        convert_to_percentiles=True,
        clip_percentiles=(5, 99),
        max_size=max_size,
        draw_topk=draw_topk,
        box_width=2,
    )

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)

    if not combine_subplots:
        heatmap_img.save(out_png, quality=92, optimize=True)
        return

    rows = math.ceil(draw_topk / 5)
    cols = min(draw_topk, 5)
    wsi = openslide.open_slide(slide_path)
    patch_grid, _ = mm._extract_top_patches_grid(
        wsi=wsi,
        coords_lvl0=coords_xy,
        scores=scores_raw,
        k=draw_topk,
        patch_level=patch_level,
        patch_size=patch_size,
        grid=(rows, cols),
        pad=8,
        cmap_name=cmap_name,
    )
    wsi.close()

    combo = mm._combine_subplot(
        left=heatmap_img,
        right=patch_grid,
        layout=layout,
        pad=30,
        bg=(255, 255, 255),
        title=title,
        title_height=50,
    )
    combo.save(out_png, quality=92, optimize=True)


In [ ]:
def inference_ensemble_with_attention(
    dataloader,
    models,
    out_dir,
    device="cuda",
    patch_level=0,
    draw_topk=10,
    alpha=0.6,
    cmap_name="plasma",
    combine_subplots=True,
    vis_level=-1,
    max_size=4096,
):
    out_dir = Path(out_dir)

    fold_att_dir = out_dir / "attentions"
    fold_png_dir = out_dir / "reports"
    ens_att_dir  = out_dir / "attentions_ensemble"
    ens_png_dir  = out_dir / "reports_ensemble"

    for p in [fold_att_dir, fold_png_dir, ens_att_dir, ens_png_dir]:
        p.mkdir(parents=True, exist_ok=True)

    predictions_rows = []
    ensemble_rows = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, total=len(dataloader), desc="External inference")):
            slide_id = batch["slide_id"][0]
            slide_path = batch["slide_path"][0]

            feats = batch["features"]
            if isinstance(feats, (list, tuple)):
                feats = feats[0]
            if feats.dim() == 3:
                feats = feats.squeeze(0)
            x = feats.to(device)  # (N,D)

            coords_xy = batch["coords_xy"]
            if isinstance(coords_xy, (list, tuple)):
                coords_xy = coords_xy[0]
            if torch.is_tensor(coords_xy):
                if coords_xy.dim() == 3:
                    coords_xy = coords_xy.squeeze(0)
                coords_xy_np = coords_xy.cpu().numpy()
            else:
                coords_xy_np = np.asarray(coords_xy)

            patch_size_lv0 = batch["patch_size_lv0"]
            if isinstance(patch_size_lv0, (list, tuple)):
                patch_size_lv0 = patch_size_lv0[0]
            if torch.is_tensor(patch_size_lv0):
                patch_size_lv0 = int(patch_size_lv0.item())
            else:
                patch_size_lv0 = int(patch_size_lv0)

            if x.shape[0] != coords_xy_np.shape[0]:
                raise RuntimeError(
                    f"{slide_id}: mismatch between features ({x.shape[0]}) and coords ({coords_xy_np.shape[0]})"
                )

            fold_raws = []
            fold_softs = []
            fold_logits = []

            for fold_idx, model in enumerate(models, start=1):
                logits, out = model(x)

                att_raw = out["attention_raw"].detach().cpu().squeeze().numpy()       # (N,)
                att_soft = out["attention_softmax"].detach().cpu().squeeze().numpy()  # (N,)

                # debug only first slide / first fold
                if batch_idx == 0 and fold_idx == 1:
                    print("\n=== ATTENTION DEBUG (FIRST SLIDE, FOLD 1) ===")
                    print("slide_id:", slide_id)
                    print("N patches:", len(att_raw))

                    print("\n--- RAW ATTENTION ---")
                    print(f"min : {att_raw.min():.6g}")
                    print(f"max : {att_raw.max():.6g}")
                    print(f"mean: {att_raw.mean():.6g}")
                    print(f"std : {att_raw.std():.6g}")
                    print("percentiles:", np.percentile(att_raw, [0, 1, 5, 50, 95, 99, 100]))

                    print("\n--- SOFTMAX ATTENTION ---")
                    print(f"min : {att_soft.min():.6g}")
                    print(f"max : {att_soft.max():.6g}")
                    print(f"mean: {att_soft.mean():.6g}")
                    print(f"std : {att_soft.std():.6g}")
                    print("percentiles:", np.percentile(att_soft, [0, 1, 5, 50, 95, 99, 100]))

                    print("\n--- LOGITS ---")
                    print(logits.detach().cpu().numpy())
                    print("============================================\n")

                # save fold npz
                fold_npz_dir = fold_att_dir / f"fold_{fold_idx}"
                fold_npz_dir.mkdir(parents=True, exist_ok=True)
                npz_path = fold_npz_dir / f"{slide_id}_att_with_coords.npz"
                save_attention_npz(
                    npz_path=npz_path,
                    coords_xy=coords_xy_np,
                    att_raw_1d=att_raw,
                    att_soft_1d=att_soft,
                    patch_size_lv0=patch_size_lv0
                )

                # save fold PNG report (RAW attention)
                fold_rep_dir = fold_png_dir / f"fold_{fold_idx}"
                fold_rep_dir.mkdir(parents=True, exist_ok=True)
                render_clam_like_report(
                    slide_path=slide_path,
                    coords_xy=coords_xy_np,
                    scores_raw=att_raw,
                    out_png=fold_rep_dir / f"{slide_id}.png",
                    patch_level=patch_level,
                    patch_size=patch_size_lv0,
                    draw_topk=draw_topk,
                    alpha=alpha,
                    cmap_name=cmap_name,
                    combine_subplots=combine_subplots,
                    vis_level=vis_level,
                    max_size=max_size,
                    layout="horizontal",
                    title=f"{slide_id} | fold {fold_idx}"
                )

                row = {
                    "slide_id": slide_id,
                    "slide_path": slide_path,
                    "fold": fold_idx,
                }
                row.update(logits_to_outputs(logits))
                predictions_rows.append(row)

                fold_raws.append(att_raw)
                fold_softs.append(att_soft)
                fold_logits.append(logits.detach().cpu().squeeze().numpy())

            # ensemble: mean of fold outputs
            att_raw_ens = np.mean(np.stack(fold_raws, axis=0), axis=0)
            att_soft_ens = np.mean(np.stack(fold_softs, axis=0), axis=0)
            logits_ens = np.mean(np.stack(fold_logits, axis=0), axis=0)

            ens_npz_path = ens_att_dir / f"{slide_id}_att_with_coords.npz"
            save_attention_npz(
                npz_path=ens_npz_path,
                coords_xy=coords_xy_np,
                att_raw_1d=att_raw_ens,
                att_soft_1d=att_soft_ens,
                patch_size_lv0=patch_size_lv0
            )

            render_clam_like_report(
                slide_path=slide_path,
                coords_xy=coords_xy_np,
                scores_raw=att_raw_ens,
                out_png=ens_png_dir / f"{slide_id}.png",
                patch_level=patch_level,
                patch_size=patch_size_lv0,
                draw_topk=draw_topk,
                alpha=alpha,
                cmap_name=cmap_name,
                combine_subplots=combine_subplots,
                vis_level=vis_level,
                max_size=max_size,
                layout="horizontal",
                title=f"{slide_id} | ensemble"
            )

            # ensemble row for CSV
            if logits_ens.ndim == 0:
                logits_ens_t = torch.tensor([[float(logits_ens)]], dtype=torch.float32)
            elif logits_ens.ndim == 1:
                logits_ens_t = torch.tensor(logits_ens[None, :], dtype=torch.float32)
            else:
                logits_ens_t = torch.tensor(logits_ens, dtype=torch.float32)

            ens_row = {
                "slide_id": slide_id,
                "slide_path": slide_path,
            }
            ens_row.update(logits_to_outputs(logits_ens_t))
            ensemble_rows.append(ens_row)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # save CSVs
    pred_df = pd.DataFrame(predictions_rows)
    ens_df = pd.DataFrame(ensemble_rows)

    pred_df.to_csv(out_dir / "predictions_per_fold.csv", index=False)
    ens_df.to_csv(out_dir / "predictions_ensemble.csv", index=False)

    print("\nSaved:")
    print(" -", out_dir / "predictions_per_fold.csv")
    print(" -", out_dir / "predictions_ensemble.csv")
    print(" - fold attentions in:", fold_att_dir)
    print(" - fold reports in   :", fold_png_dir)
    print(" - ensemble attentions:", ens_att_dir)
    print(" - ensemble reports   :", ens_png_dir)


In [24]:
def run_fold_inference_and_save_attention(
    dataloader,
    models,
    out_dir,
    device="cuda",
):
    """
    Runs inference for every slide with every fold model.
    Saves:
      - fold attention npz
      - predictions_per_fold.csv

    Does NOT generate PNG reports.
    """
    out_dir = Path(out_dir)
    fold_att_dir = out_dir / "attentions"
    fold_att_dir.mkdir(parents=True, exist_ok=True)

    predictions_rows = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, total=len(dataloader), desc="Inference only")):
            slide_id = batch["slide_id"][0]
            slide_path = batch["slide_path"][0]

            feats = batch["features"]
            if isinstance(feats, (list, tuple)):
                feats = feats[0]
            if feats.dim() == 3:
                feats = feats.squeeze(0)
            x = feats.to(device)  # (N,D)

            coords_xy = batch["coords_xy"]
            if isinstance(coords_xy, (list, tuple)):
                coords_xy = coords_xy[0]
            if torch.is_tensor(coords_xy):
                if coords_xy.dim() == 3:
                    coords_xy = coords_xy.squeeze(0)
                coords_xy_np = coords_xy.cpu().numpy()
            else:
                coords_xy_np = np.asarray(coords_xy)

            patch_size_lv0 = batch["patch_size_lv0"]
            if isinstance(patch_size_lv0, (list, tuple)):
                patch_size_lv0 = patch_size_lv0[0]
            if torch.is_tensor(patch_size_lv0):
                patch_size_lv0 = int(patch_size_lv0.item())
            else:
                patch_size_lv0 = int(patch_size_lv0)

            if x.shape[0] != coords_xy_np.shape[0]:
                raise RuntimeError(
                    f"{slide_id}: mismatch between features ({x.shape[0]}) and coords ({coords_xy_np.shape[0]})"
                )

            for fold_idx, model in enumerate(models, start=1):
                logits, out = model(x)

                att_raw = out["attention_raw"].detach().cpu().squeeze().numpy()       # (N,)
                att_soft = out["attention_softmax"].detach().cpu().squeeze().numpy()  # (N,)

                # debug only first slide / first fold
                if batch_idx == 0 and fold_idx == 1:
                    print("\n=== ATTENTION DEBUG (FIRST SLIDE, FOLD 1) ===")
                    print("slide_id:", slide_id)
                    print("N patches:", len(att_raw))

                    print("\n--- RAW ATTENTION ---")
                    print(f"min : {att_raw.min():.6g}")
                    print(f"max : {att_raw.max():.6g}")
                    print(f"mean: {att_raw.mean():.6g}")
                    print(f"std : {att_raw.std():.6g}")
                    print("percentiles:", np.percentile(att_raw, [0, 1, 5, 50, 95, 99, 100]))

                    print("\n--- SOFTMAX ATTENTION ---")
                    print(f"min : {att_soft.min():.6g}")
                    print(f"max : {att_soft.max():.6g}")
                    print(f"mean: {att_soft.mean():.6g}")
                    print(f"std : {att_soft.std():.6g}")
                    print("percentiles:", np.percentile(att_soft, [0, 1, 5, 50, 95, 99, 100]))

                    print("\n--- LOGITS ---")
                    print(logits.detach().cpu().numpy())
                    print("============================================\n")

                # save fold npz
                fold_npz_dir = fold_att_dir / f"fold_{fold_idx}"
                fold_npz_dir.mkdir(parents=True, exist_ok=True)
                npz_path = fold_npz_dir / f"{slide_id}_att_with_coords.npz"

                save_attention_npz(
                    npz_path=npz_path,
                    coords_xy=coords_xy_np,
                    att_raw_1d=att_raw,
                    att_soft_1d=att_soft,
                    patch_size_lv0=patch_size_lv0
                )

                row = {
                    "slide_id": slide_id,
                    "slide_path": slide_path,
                    "fold": fold_idx,
                }
                row.update(logits_to_outputs(logits))
                predictions_rows.append(row)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    pred_df = pd.DataFrame(predictions_rows)
    pred_csv = out_dir / "predictions_per_fold.csv"
    pred_df.to_csv(pred_csv, index=False)

    print("\nSaved inference outputs:")
    print(" -", pred_csv)
    print(" - fold attentions in:", fold_att_dir)

In [25]:
def build_ensemble_attention(
    dataloader,
    out_dir,
    n_folds=5,
):
    """
    Reads fold attention npz files already saved on disk.
    Builds:
      - ensemble attention npz
      - predictions_ensemble.csv

    Does NOT generate PNG reports.
    """
    out_dir = Path(out_dir)
    fold_att_dir = out_dir / "attentions"
    ens_att_dir = out_dir / "attentions_ensemble"
    ens_att_dir.mkdir(parents=True, exist_ok=True)

    pred_df = pd.read_csv(out_dir / "predictions_per_fold.csv")
    ensemble_rows = []

    for batch in tqdm(dataloader, total=len(dataloader), desc="Build ensemble only"):
        slide_id = batch["slide_id"][0]
        slide_path = batch["slide_path"][0]

        coords_xy = batch["coords_xy"]
        if isinstance(coords_xy, (list, tuple)):
            coords_xy = coords_xy[0]
        if torch.is_tensor(coords_xy):
            if coords_xy.dim() == 3:
                coords_xy = coords_xy.squeeze(0)
            coords_xy_np = coords_xy.cpu().numpy()
        else:
            coords_xy_np = np.asarray(coords_xy)

        patch_size_lv0 = batch["patch_size_lv0"]
        if isinstance(patch_size_lv0, (list, tuple)):
            patch_size_lv0 = patch_size_lv0[0]
        if torch.is_tensor(patch_size_lv0):
            patch_size_lv0 = int(patch_size_lv0.item())
        else:
            patch_size_lv0 = int(patch_size_lv0)

        fold_raws = []
        fold_softs = []

        for fold_idx in range(1, n_folds + 1):
            npz_path = fold_att_dir / f"fold_{fold_idx}" / f"{slide_id}_att_with_coords.npz"
            if not npz_path.exists():
                raise FileNotFoundError(npz_path)

            data = np.load(npz_path)
            fold_raws.append(data["attention_raw"])
            fold_softs.append(data["attention"])
            data.close()

        att_raw_ens = np.mean(np.stack(fold_raws, axis=0), axis=0)
        att_soft_ens = np.mean(np.stack(fold_softs, axis=0), axis=0)

        ens_npz_path = ens_att_dir / f"{slide_id}_att_with_coords.npz"
        save_attention_npz(
            npz_path=ens_npz_path,
            coords_xy=coords_xy_np,
            att_raw_1d=att_raw_ens,
            att_soft_1d=att_soft_ens,
            patch_size_lv0=patch_size_lv0
        )

        # ensemble predictions from predictions_per_fold.csv
        slide_pred_df = pred_df[pred_df["slide_id"] == slide_id].copy()
        if len(slide_pred_df) != n_folds:
            raise RuntimeError(f"{slide_id}: expected {n_folds} rows in predictions_per_fold.csv, got {len(slide_pred_df)}")

        ens_row = {
            "slide_id": slide_id,
            "slide_path": slide_path,
        }

        # average available prediction columns
        pred_cols = [c for c in slide_pred_df.columns if c.startswith("prob_") or c.startswith("logit_")]
        for col in pred_cols:
            ens_row[col] = float(slide_pred_df[col].mean())

        # final pred class
        if "prob_1" in ens_row:
            ens_row["pred_class"] = int(ens_row["prob_1"] >= 0.5)
        else:
            # fallback if only logit_1 exists
            if "logit_1" in ens_row:
                ens_row["pred_class"] = int(ens_row["logit_1"] >= 0.0)

        ensemble_rows.append(ens_row)

    ens_df = pd.DataFrame(ensemble_rows)
    ens_csv = out_dir / "predictions_ensemble.csv"
    ens_df.to_csv(ens_csv, index=False)

    print("\nSaved ensemble outputs:")
    print(" -", ens_csv)
    print(" - ensemble attentions in:", ens_att_dir)

In [ ]:
def generate_reports_from_attention_dir(
    attention_dir,
    dataloader,
    output_dir,
    use_raw=True,
    patch_level=0,
    draw_topk=10,
    alpha=0.6,
    cmap_name="plasma",
    combine_subplots=True,
    vis_level=-1,
    max_size=4096,
):
    """
    Reads attention npz files from disk and generates PNG reports only.
    No inference is performed here.

    combine_subplots=False  → heatmap-only output, suitable for paper figures.
    """
    attention_dir = Path(attention_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for batch in tqdm(dataloader, total=len(dataloader), desc=f"Generate reports from {attention_dir.name}"):
        slide_id = batch["slide_id"][0]
        slide_path = batch["slide_path"][0]

        npz_path = attention_dir / f"{slide_id}_att_with_coords.npz"
        if not npz_path.exists():
            raise FileNotFoundError(npz_path)

        data = np.load(npz_path)
        coords_xy = data["coords"]

        if "patch_size_lv0" in data:
            patch_size_arr = data["patch_size_lv0"]
            patch_size_lv0 = int(patch_size_arr[0]) if np.ndim(patch_size_arr) > 0 else int(patch_size_arr)
        else:
            patch_size_lv0 = DEFAULT_PATCH_SIZE

        if use_raw:
            if "attention_raw" not in data:
                raise KeyError(f"{npz_path} does not contain attention_raw")
            scores_raw = data["attention_raw"].astype(np.float32).reshape(-1)
        else:
            scores_raw = data["attention"].astype(np.float32).reshape(-1)

        out_png = output_dir / f"{slide_id}.png"

        render_clam_like_report(
            slide_path=slide_path,
            coords_xy=coords_xy,
            scores_raw=scores_raw,
            out_png=out_png,
            patch_level=patch_level,
            patch_size=patch_size_lv0,
            draw_topk=draw_topk,
            alpha=alpha,
            cmap_name=cmap_name,
            combine_subplots=combine_subplots,
            vis_level=vis_level,
            max_size=max_size,
            layout="horizontal",
            title=f"{slide_id}",
        )

        data.close()
        gc.collect()

    print(f"\nSaved reports to: {output_dir}")


In [17]:
run_fold_inference_and_save_attention(
    dataloader=dl,
    models=models,
    out_dir=OUT_DIR,
    device=DEVICE,
)

Inference only:   0%|          | 0/44 [00:00<?, ?it/s]


=== ATTENTION DEBUG (FIRST SLIDE, FOLD 1) ===
slide_id: PAN_CNI_S000201
N patches: 102297

--- RAW ATTENTION ---
min : -1.35474
max : 3.67828
mean: 0.832542
std : 0.912862
percentiles: [-1.35473788 -0.78687606 -0.47057248  0.72935253  2.46273475  2.98224806
  3.67827678]

--- SOFTMAX ATTENTION ---
min : 7.04527e-07
max : 0.000108071
mean: 9.77546e-06
std : 1.08794e-05
percentiles: [7.04526997e-07 1.24313111e-06 1.70563410e-06 5.66247945e-06
 3.20482999e-05 5.38798972e-05 1.08070752e-04]

--- LOGITS ---
[[2.2397149]]



Inference only: 100%|██████████| 44/44 [00:28<00:00,  1.56it/s]


Saved inference outputs:
 - /opt/app/user/cnio_attention_zones/predictions_per_fold.csv
 - fold attentions in: /opt/app/user/cnio_attention_zones/attentions


In [18]:
build_ensemble_attention(
    dataloader=dl,
    out_dir=OUT_DIR,
    n_folds=len(models),
)

Build ensemble only: 100%|██████████| 44/44 [00:13<00:00,  3.25it/s]


Saved ensemble outputs:
 - /opt/app/user/cnio_attention_zones/predictions_ensemble.csv
 - ensemble attentions in: /opt/app/user/cnio_attention_zones/attentions_ensemble


In [12]:
generate_reports_from_attention_dir(
    attention_dir=Path(OUT_DIR) / "attentions_ensemble",
    dataloader=dl,
    output_dir=Path(OUT_DIR) / "reports_ensemble",
    use_raw=True,
    patch_level=PATCH_LEVEL,
    draw_topk=DRAW_TOPK,
    alpha=ALPHA,
    vis_level=VIS_LEVEL,
    max_size=MAX_SIZE,
)

Generate reports from attentions_ensemble:   0%|          | 0/47 [00:30<?, ?it/s]


NameError: name 'extract_top_patches_grid' is not defined

In [27]:
for fold_idx in range(1, len(models) + 1):
    generate_reports_from_attention_dir(
        attention_dir=Path(OUT_DIR) / "attentions" / f"fold_{fold_idx}",
        dataloader=dl,
        output_dir=Path(OUT_DIR) / "reports" / f"fold_{fold_idx}",
        use_raw=True,
        patch_level=PATCH_LEVEL,
        draw_topk=DRAW_TOPK,
        alpha=ALPHA,
        vis_level=VIS_LEVEL,
        max_size=MAX_SIZE,
    )

Generate reports from fold_1:  94%|█████████▎| 44/47 [24:36<01:40, 33.57s/it]


FileNotFoundError: /opt/app/user/cnio_attention_zones/attentions/fold_1/PAN_CNI_S000173_att_with_coords.npz

In [12]:
sample = ds[0]
print("slide_id:", sample["slide_id"])
print("slide_path:", sample["slide_path"])
print("features shape:", tuple(sample["features"].shape))
print("coords shape:", sample["coords_xy"].shape)
print("patch_size_lv0:", sample["patch_size_lv0"])

assert sample["features"].shape[0] == sample["coords_xy"].shape[0]
print("Sanity check passed.")

slide_id: PAN_CNI_S000201
slide_path: /opt/app/user/zoomed/PAN_CNI_S000201.tif
features shape: (102297, 1024)
coords shape: (102297, 2)
patch_size_lv0: 224
Sanity check passed.


In [13]:
inference_ensemble_with_attention(
    dataloader=dl,
    models=models,
    out_dir=OUT_DIR,
    device=DEVICE,
    patch_level=PATCH_LEVEL,
    draw_topk=DRAW_TOPK,
    alpha=ALPHA,
    vis_level=VIS_LEVEL,
    max_size=MAX_SIZE,
)

External inference:   0%|          | 0/44 [00:00<?, ?it/s]


=== ATTENTION DEBUG (FIRST SLIDE, FOLD 1) ===
slide_id: PAN_CNI_S000201
N patches: 102297

--- RAW ATTENTION ---
min : -1.35474
max : 3.67828
mean: 0.832542
std : 0.912862
percentiles: [-1.35473788 -0.78687606 -0.47057248  0.72935253  2.46273475  2.98224806
  3.67827678]

--- SOFTMAX ATTENTION ---
min : 7.04527e-07
max : 0.000108071
mean: 9.77546e-06
std : 1.08794e-05
percentiles: [7.04526997e-07 1.24313111e-06 1.70563410e-06 5.66247945e-06
 3.20482999e-05 5.38798972e-05 1.08070752e-04]

--- LOGITS ---
[[2.2397149]]



External inference:   9%|▉         | 4/44 [10:03<1:40:34, 150.86s/it]

KeyboardInterrupt



In [ ]:
print("OUT_DIR:", OUT_DIR)
print("\nExpected structure:")
print(f"""
{OUT_DIR}/
  predictions_per_fold.csv
  predictions_ensemble.csv
  attentions/
    fold_1/
    fold_2/
    fold_3/
    fold_4/
    fold_5/
  reports/
    fold_1/
    fold_2/
    fold_3/
    fold_4/
    fold_5/
  attentions_ensemble/
  reports_ensemble/
""")